# EarthBridge Colab Baseline Training

Use this notebook as a backup to Kaggle. Train online, export artifacts, then run the final demo locally from the downloaded zip.

In [ ]:
from pathlib import Path
import os
import subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Not running in Colab or Drive mount skipped.')

In [ ]:
REPO_URL = ""  # Optional: set to your GitHub repo URL.
ROOT = Path("/content/EarthBridge")

if REPO_URL and not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)

if not ROOT.exists():
    ROOT = Path.cwd()

os.chdir(ROOT)
print("Working directory:", ROOT)

In [ ]:
!python -m pip install -q -r requirements.txt
!python -m pip install -q faiss-cpu rasterio albumentations opencv-python matplotlib tensorboard fastapi "uvicorn[standard]" python-multipart
!python -m pip install -q -e .

## Configure Dataset

Put a smaller dataset or subset in Drive, then set `DATA_RAW` to that raw dataset directory.

In [ ]:
DATA_RAW = Path("/content/drive/MyDrive/earthbridge/data/raw")
IMAGE_ROOT = DATA_RAW.parent
LEFT_MODALITY = "optical_rgb"
RIGHT_MODALITY = "sar"

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("DATA_RAW:", DATA_RAW)
print("IMAGE_ROOT:", IMAGE_ROOT)
print("PAIR:", LEFT_MODALITY, "->", RIGHT_MODALITY)
print("DEVICE:", DEVICE)
assert DATA_RAW.exists(), f"Dataset path does not exist: {DATA_RAW}"

In [ ]:
!python scripts/run_cloud_pipeline.py \
  --data-raw "{DATA_RAW}" \
  --image-root "{IMAGE_ROOT}" \
  --left-modality "{LEFT_MODALITY}" \
  --right-modality "{RIGHT_MODALITY}" \
  --image-size 128 \
  --batch-size 16 \
  --epochs 5 \
  --device "{DEVICE}" \
  --top-k 10 \
  --allow-missing-labels \
  --export-zip artifacts/earthbridge_export.zip
print("Download this file:", ROOT / "artifacts/earthbridge_export.zip")